# Location Extraction: Span-Based NER with BERT Models

This notebook compares **6 different approaches** for extracting location information from conflict event descriptions using span-based Named Entity Recognition (NER):

1. **GLiNER (zero-shot)** - Modern zero-shot NER baseline with slot-filling
2. **ConfliBERT** - BERT pretrained on conflict-related text
3. **DeBERTa-v3** - Enhanced BERT with disentangled attention
4. **XLM-RoBERTa** - Multilingual BERT model
5. **SpanBERT** - BERT optimized for span-based tasks
6. **MuRIL** - Multilingual Representations for Indian Languages

**Dataset**: Armed Assault + Bombing incidents with human-annotated location data

**Approach**: Train span-NER models to extract and type location entities (`STATE`, `DISTRICT`, `VILLAGE`, `OTHER_LOCATION`), then slot-fill to structured format.

## Notebook Structure

1. **Setup & Configuration** - Environment setup and library imports
2. **Data Handling** - Load data, generate silver span annotations, temporal split
3. **Model Setup** - Configure GLiNER baseline and BERT models
4. **Model Training & Evaluation** - Train and evaluate each model
5. **Results & Comparison** - Compare models and analyze performance


#### About Span-Based NER Approach

This notebook implements a **span-based NER pipeline** for location extraction:

1. **Silver Span Generation**
   - Auto-generate character-offset annotations from structured columns (`state`, `district`, `village_name`, `other_locations`)
   - Use exact → normalized → fuzzy matching to align text spans
   - Entity types: `STATE`, `DISTRICT`, `VILLAGE`, `OTHER_LOCATION`

2. **Model Training**
   - Train BERT models with token classification heads for NER
   - Use BIO tagging scheme for entity boundaries
   - Handle class imbalance with weighted loss

3. **Inference & Slot-Filling**
   - Extract entity spans from predictions
   - Apply Non-Maximum Suppression (NMS) for overlaps
   - Slot-fill: top-1 for `STATE`/`DISTRICT`/`VILLAGE`, collect all `OTHER_LOCATION`
   - Convert to structured format: `"state: X, district: Y, village: Z, other_locations: A, B"`

**Potential benefits over seq2seq:**
- Better boundary detection (character-level precision)
- Fewer formatting errors (structured by design)
- Explicit entity typing (learns distinction between state/district/village)


#### About Data Splits

- **Temporal Split (80/10/10):**
  - Training on earlier data, testing on more recent data (realistic deployment scenario)
  - Training ends before Telangana formation (June 2, 2014)
  - Test set contains only post-2014 incidents with current state names
- **Reasoning:**
  - Temporal split better simulates real-world deployment scenario
  - Keeps recent data for evaluation (important for model performance)
  - Avoids data leakage between train/val/test sets


#### Metrics

- **Training Metrics:**
  - **Token classification loss** on validation set
  - Early stopping based on validation loss
  
- **Evaluation Metrics:**
  - **Exact Match Accuracy**: All levels must match perfectly
  - **Fuzzy Match Accuracy**: Using `rapidfuzz` with 85% threshold
  - **Per-Level Metrics**: Precision, Recall, F1 for each of state/district/village/other_locations
  - **Micro-averaged F1**: Overall performance across all levels
  
- **Compatibility:**
  - All metrics compatible with seq2seq models for direct comparison
  - Uses same `metrics_utils.py` functions


## 1. Setup and Configuration


### 1.1 Environment Setup


#### 1.1.1 Colab Setup


In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2) Clone or update the repo
BRANCH = "main"  # Change if working on a different branch

# Clone fresh each session 
!rm -rf /content/code-satp
!git clone -b $BRANCH --depth 1 https://github.com/eteitelbaum/code-satp.git /content/code-satp

# 3) Install required packages
%pip install -qU pip setuptools wheel
%pip install --upgrade transformers datasets evaluate seqeval gliner rapidfuzz accelerate

# 4) Make result directories
import pathlib
import sys
pathlib.Path("/content/drive/MyDrive/colab/satp-results").mkdir(parents=True, exist_ok=True)
pathlib.Path("/content/drive/MyDrive/colab/satp-results/location-extraction-bert").mkdir(parents=True, exist_ok=True)

# 5) Import utilities from cloned repo
try:
    import importlib.util
    
    # Import file_io utilities
    spec = importlib.util.spec_from_file_location(
        "file_io",
        "/content/code-satp/models/classification-models/utils/file_io.py"
    )
    file_io = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(file_io)
    
    get_task_results_dir = file_io.get_task_results_dir
    save_dataframe_csv = file_io.save_dataframe_csv
    
    # Add location-models to path FIRST to prioritize local utilities
    sys.path.insert(0, "/content/code-satp/models/location-models")
    
    # Define task name for consistent file organization
    TASK_NAME = "location-extraction-bert"
    
    print("="*80)
    print("✅ SETUP COMPLETE")
    print("="*80)
    print(f"📂 Cloned repo: /content/code-satp")
    print(f"📂 Results dir: {get_task_results_dir(TASK_NAME)}")
    print(f"✅ File I/O utilities loaded successfully")
    print("="*80)
    
except Exception as e:
    print("="*80)
    print("⚠️  WARNING: Could not load file_io utilities")
    print(f"Error: {e}")
    print("="*80)
    print("Falling back to local mode - files will be saved to current directory")
    
    # Define fallback functions
    TASK_NAME = "location-extraction-bert"
    
    def get_task_results_dir(task_name):
        return pathlib.Path(f"./results/{task_name}")
    
    def save_dataframe_csv(df, filename, task_name=None):
        output_path = f"./results/{task_name}/{filename}" if task_name else f"./{filename}"
        pathlib.Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_path, index=False)
        return output_path
    
    print("="*80)


#### 1.1.2 Local Setup


In [ ]:
# # For local development, uncomment these lines:
# import sys
# import os

# # Local data and results paths
# DATA_PATH = "../../data/"
# RESULTS_PATH = "./results/"

# # Create local results directory
# os.makedirs(RESULTS_PATH, exist_ok=True)

# # Verify GPU availability
# import torch
# if torch.cuda.is_available():
#     print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
# else:
#     print("⚠️ GPU not available, using CPU")

# print("✅ Local setup complete!")


### 1.2 Import Libraries


In [ ]:
# Core libraries
import os
import re
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

# Machine learning and transformers
import torch
from torch.utils.data import Dataset as TorchDataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from datasets import Dataset as HFDataset

# GLiNER for zero-shot baseline
from gliner import GLiNER

# Evaluation metrics
from sklearn.model_selection import train_test_split

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Progress bars
from tqdm import tqdm

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Import local utilities
from utils.span_utils import (
    normalize_text,
    align_location_to_spans,
    create_ner_dataset,
    validate_ner_data,
)
from utils.gliner_utils import (
    LOCATION_LABELS,
    predict_entities_gliner,
    run_gliner_extraction,
    batch_extract_locations,
    save_gliner_predictions_and_metrics,
)
from utils.span_ner_utils import (
    get_label_list,
    get_id_to_label_mapping,
    tokenize_and_align_labels,
    predictions_to_entities,
    non_maximum_suppression,
    slot_fill_from_entities,
    slots_to_structured_string,
)
from utils.bert_model_utils import (
    BERT_MODEL_CONFIGS,
    run_span_ner_model,
    evaluate_ner_model,
    save_bert_predictions_and_metrics,
)
from utils.data_utils import (
    build_structured_location,
    preview_location_examples,
)
from utils.metrics_utils import (
    compute_metrics,
    print_metrics,
    parse_structured_location,
    fuzzy_match,
)

print("✅ All libraries imported successfully!")


## 2. Data Handling


### 2.1 Load Raw Data


In [ ]:
# Load location data
data_path = "/content/code-satp/data/location_info_augmented.csv"  # Colab path
# data_path = "../../data/location_info_augmented.csv"  # Uncomment for local

df = pd.read_csv(data_path)

print(f"✅ Loaded {len(df):,} incidents")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

# Display sample
df.head()


### 2.2 Data Preprocessing


In [ ]:
# Clean and prepare data
df['date'] = pd.to_datetime(df['date'])

# Remove incidents with missing text
df = df[df['incident_summary'].notna()].copy()
df = df[df['incident_summary'].str.strip() != ''].copy()

# Sort by date for temporal split
df = df.sort_values('date').reset_index(drop=True)

print(f"✅ After cleaning: {len(df):,} incidents")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")


### 2.3 Generate Silver Span Annotations

Auto-generate character-offset span annotations by aligning location names from structured columns back to the incident text.


In [ ]:
print("Generating silver span annotations...")
print("This may take a few minutes...\n")

# Create NER dataset with span annotations
ner_data = create_ner_dataset(df, text_col='incident_summary')

print(f"\n✅ Generated {len(ner_data):,} NER examples")

# Validate and show statistics
stats = validate_ner_data(ner_data)

print("\n" + "="*60)
print("SILVER SPAN ANNOTATION STATISTICS")
print("="*60)
print(f"Total examples: {stats['total_examples']:,}")
print(f"Examples with entities: {stats['examples_with_entities']:,} ({stats['examples_with_entities']/stats['total_examples']*100:.1f}%)")
print(f"Examples without entities: {stats['examples_without_entities']:,}")
print(f"\nTotal entities: {stats['total_entities']:,}")
print(f"Avg entities per example: {stats['avg_entities_per_example']:.2f}")
print(f"\nEntities by label:")
for label, count in sorted(stats['entities_by_label'].items()):
    print(f"  {label}: {count:,}")
print(f"\nExamples with STATE+DISTRICT+VILLAGE: {stats['examples_with_all_types']:,}")
print("="*60)


### 2.4 Preview Generated Spans


In [ ]:
# Show examples of generated spans
print("\n" + "="*80)
print("SAMPLE NER ANNOTATIONS")
print("="*80)

for i in range(min(3, len(ner_data))):
    example = ner_data[i]
    text = example['text']
    entities = example['entities']
    
    print(f"\n[Example {i+1}]")
    print(f"Text: {text[:200]}...") if len(text) > 200 else print(f"Text: {text}")
    print(f"\nEntities ({len(entities)}):")
    for ent in entities:
        print(f"  - {ent['label']:15s} | {ent['text']:30s} | chars [{ent['start']:3d}, {ent['end']:3d}]")
    print("-" * 80)


### 2.5 Temporal Data Split (80/10/10)

Split data temporally: earlier data for training, recent data for testing.


In [ ]:
# Extract dates for temporal split
dates = [pd.to_datetime(ex['metadata']['date']) for ex in ner_data]

# Calculate split indices
n = len(ner_data)
train_idx = int(0.8 * n)
val_idx = int(0.9 * n)

# Split data
train_data = ner_data[:train_idx]
val_data = ner_data[train_idx:val_idx]
test_data = ner_data[val_idx:]

print("\n" + "="*60)
print("TEMPORAL DATA SPLIT")
print("="*60)
print(f"Train: {len(train_data):,} examples ({len(train_data)/n*100:.1f}%)")
print(f"  Date range: {dates[0]} to {dates[train_idx-1]}")
print(f"\nVal:   {len(val_data):,} examples ({len(val_data)/n*100:.1f}%)")
print(f"  Date range: {dates[train_idx]} to {dates[val_idx-1]}")
print(f"\nTest:  {len(test_data):,} examples ({len(test_data)/n*100:.1f}%)")
print(f"  Date range: {dates[val_idx]} to {dates[-1]}")
print("="*60)

# Validate splits
for split_name, split_data in [("Train", train_data), ("Val", val_data), ("Test", test_data)]:
    stats = validate_ner_data(split_data)
    print(f"\n{split_name} set: {stats['examples_with_entities']}/{stats['total_examples']} examples with entities")


### 2.6 Prepare Ground Truth for Evaluation

Add ground truth location metadata to test data for evaluation.


In [ ]:
# Add ground truth metadata to test examples
# This is needed for evaluation using existing metrics

# Get original rows for test set
test_incident_numbers = [ex['metadata']['incident_number'] for ex in test_data]

for i, ex in enumerate(test_data):
    incident_num = ex['metadata']['incident_number']
    # Find matching row in original dataframe
    matching_rows = df[df['incident_number'] == incident_num]
    
    if len(matching_rows) > 0:
        row = matching_rows.iloc[0]
        # Add ground truth location fields
        ex['metadata']['state'] = str(row.get('state', ''))
        ex['metadata']['district'] = str(row.get('district', ''))
        ex['metadata']['village_name'] = str(row.get('village_name', ''))
        ex['metadata']['other_locations'] = str(row.get('other_locations', ''))

print("✅ Ground truth metadata added to test set")


## 3. Model Setup


### 3.1 Configure Models

We'll train 5 BERT-based models and compare them to a GLiNER zero-shot baseline:


In [ ]:
# Display model configurations
print("\n" + "="*80)
print("MODEL CONFIGURATIONS")
print("="*80)

print("\n1. GLiNER (Zero-Shot Baseline)")
print("   Model: urchade/gliner_multi-v2.1")
print("   Description: Multilingual zero-shot NER with slot-filling")

for i, (key, config) in enumerate(BERT_MODEL_CONFIGS.items(), 2):
    print(f"\n{i}. {key}")
    print(f"   Model: {config['model_name']}")
    print(f"   Description: {config['description']}")

print("\n" + "="*80)


### 3.2 Training Configuration


In [ ]:
# Training hyperparameters
TRAINING_CONFIG = {
    'num_epochs': 3,
    'batch_size': 16,
    'learning_rate': 5e-5,
    'max_length': 512,
}

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\nTraining configuration:")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")


### 3.3 Results Storage Setup


In [ ]:
# Create directories for results storage
results_dir = get_task_results_dir(TASK_NAME)
results_dir.mkdir(parents=True, exist_ok=True)

# Dictionary to store all results
all_results = {}

print(f"✅ Results will be saved to: {results_dir}")


## 4. Model Training and Evaluation


### 4.1 Model 1: GLiNER (Zero-Shot Baseline)


In [ ]:
model_name = "gliner"

print("\n" + "="*80)
print(f"MODEL 1: GLiNER (Zero-Shot)")
print("="*80)

# Load GLiNER model
print("\nLoading GLiNER model...")
gliner_model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")
print("✅ Model loaded")

# Extract test texts
test_texts = [ex['text'] for ex in test_data]

# Run batch extraction
print("\nRunning zero-shot extraction on test set...")
gliner_predictions = batch_extract_locations(gliner_model, test_texts, show_progress=True)

# Extract structured predictions
pred_structured = [result['structured_location'] for result in gliner_predictions]

# Build ground truth
print("\nBuilding ground truth...")
ground_truth = []
for ex in test_data:
    row_dict = {
        'state': ex['metadata']['state'],
        'district': ex['metadata']['district'],
        'village_name': ex['metadata']['village_name'],
        'other_locations': ex['metadata']['other_locations'],
    }
    row = pd.Series(row_dict)
    gt_structured = build_structured_location(row)
    ground_truth.append(gt_structured)

# Compute metrics manually (since GLiNER doesn't train)
print("\nComputing metrics...")
from utils.metrics_utils import parse_structured_location, fuzzy_match

exact_matches = 0
fuzzy_matches = 0

level_metrics = {
    'state': {'tp': 0, 'fp': 0, 'fn': 0},
    'district': {'tp': 0, 'fp': 0, 'fn': 0},
    'village': {'tp': 0, 'fp': 0, 'fn': 0},
    'other_locations': {'tp': 0, 'fp': 0, 'fn': 0},
}

for pred_str, gt_str in zip(pred_structured, ground_truth):
    # Exact match
    if pred_str.strip() == gt_str.strip():
        exact_matches += 1
        fuzzy_matches += 1
    else:
        # Fuzzy match
        if fuzzy_match(pred_str, gt_str, threshold=85):
            fuzzy_matches += 1
    
    # Parse predictions and ground truth
    pred_dict = parse_structured_location(pred_str)
    gt_dict = parse_structured_location(gt_str)
    
    # Compute per-level metrics
    for level in ['state', 'district', 'village', 'other_locations']:
        pred_val = pred_dict.get(level, '')
        gt_val = gt_dict.get(level, '')
        
        if level == 'other_locations':
            pred_list = [x.strip() for x in pred_val.split(',') if x.strip()] if pred_val else []
            gt_list = [x.strip() for x in gt_val.split(',') if x.strip()] if gt_val else []
            
            pred_set = set(pred_list)
            gt_set = set(gt_list)
            
            tp = len(pred_set & gt_set)
            fp = len(pred_set - gt_set)
            fn = len(gt_set - pred_set)
        else:
            pred_clean = pred_val.strip().lower() if pred_val else ''
            gt_clean = gt_val.strip().lower() if gt_val else ''
            
            if pred_clean and gt_clean:
                if pred_clean == gt_clean:
                    tp, fp, fn = 1, 0, 0
                else:
                    tp, fp, fn = 0, 1, 1
            elif pred_clean and not gt_clean:
                tp, fp, fn = 0, 1, 0
            elif not pred_clean and gt_clean:
                tp, fp, fn = 0, 0, 1
            else:
                tp, fp, fn = 0, 0, 0
        
        level_metrics[level]['tp'] += tp
        level_metrics[level]['fp'] += fp
        level_metrics[level]['fn'] += fn

# Calculate metrics
total_examples = len(pred_structured)
exact_match_acc = exact_matches / total_examples
fuzzy_match_acc = fuzzy_matches / total_examples

per_level_metrics = {}
for level, counts in level_metrics.items():
    tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    per_level_metrics[level] = {'precision': precision, 'recall': recall, 'f1': f1}

total_tp = sum(m['tp'] for m in level_metrics.values())
total_fp = sum(m['fp'] for m in level_metrics.values())
total_fn = sum(m['fn'] for m in level_metrics.values())

micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if (micro_precision + micro_recall) > 0 else 0

metrics = {
    'exact_match': exact_match_acc,
    'fuzzy_match': fuzzy_match_acc,
    'micro_precision': micro_precision,
    'micro_recall': micro_recall,
    'micro_f1': micro_f1,
    'per_level': per_level_metrics,
}

# Print results
print("\n" + "="*60)
print("GLiNER TEST RESULTS")
print("="*60)
print(f"Exact Match:     {metrics['exact_match']:.4f}")
print(f"Fuzzy Match:     {metrics['fuzzy_match']:.4f}")
print(f"Micro Precision: {metrics['micro_precision']:.4f}")
print(f"Micro Recall:    {metrics['micro_recall']:.4f}")
print(f"Micro F1:        {metrics['micro_f1']:.4f}")
print("\nPer-Level F1:")
for level, m in metrics['per_level'].items():
    print(f"  {level:20s}: {m['f1']:.4f}")
print("="*60)

# Store results
all_results[model_name] = {
    'metrics': metrics,
    'predictions': pred_structured,
    'ground_truth': ground_truth,
}

# Save predictions and metrics using utility function
save_gliner_predictions_and_metrics(
    model_name=model_name,
    predictions=pred_structured,
    ground_truth=ground_truth,
    metrics=metrics,
    test_data=test_data,
    task_name=TASK_NAME,
    save_dataframe_csv_func=save_dataframe_csv,
)


### 4.2 Train BERT Models

Now we'll train the 5 BERT-based NER models. Each model will be trained with the same configuration and evaluated on the test set.


#### 4.2.1 ConfliBERT


In [ ]:
model_key = 'confliBERT'
output_dir = results_dir / model_key

# Run training and evaluation
results = run_span_ner_model(
    model_key=model_key,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    output_dir=str(output_dir),
    **TRAINING_CONFIG,
    device=device,
)

# Store results
all_results[model_key] = results

# Save predictions and metrics using utility function
save_bert_predictions_and_metrics(
    model_key=model_key,
    results=results,
    test_data=test_data,
    task_name=TASK_NAME,
    save_dataframe_csv_func=save_dataframe_csv,
)


### 4.2.2 DeBERTa-v3

In [ ]:
model_key = 'deberta-v3'
output_dir = results_dir / model_key

# Run training and evaluation
results = run_span_ner_model(
    model_key=model_key,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    output_dir=str(output_dir),
    **TRAINING_CONFIG,
    device=device,
)

# Store results
all_results[model_key] = results

# Save predictions and metrics using utility function
save_bert_predictions_and_metrics(
    model_key=model_key,
    results=results,
    test_data=test_data,
    task_name=TASK_NAME,
    save_dataframe_csv_func=save_dataframe_csv,
)


#### 4.2.3 XLM-RoBERTa


In [ ]:
model_key = 'xlm-roberta'
output_dir = results_dir / model_key

# Run training and evaluation
results = run_span_ner_model(
    model_key=model_key,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    output_dir=str(output_dir),
    **TRAINING_CONFIG,
    device=device,
)

# Store results
all_results[model_key] = results

# Save predictions and metrics using utility function
save_bert_predictions_and_metrics(
    model_key=model_key,
    results=results,
    test_data=test_data,
    task_name=TASK_NAME,
    save_dataframe_csv_func=save_dataframe_csv,
)


#### 4.2.4 SpanBERT


In [ ]:
model_key = 'spanbert'
output_dir = results_dir / model_key

# Run training and evaluation
results = run_span_ner_model(
    model_key=model_key,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    output_dir=str(output_dir),
    **TRAINING_CONFIG,
    device=device,
)

# Store results
all_results[model_key] = results

# Save predictions and metrics using utility function
save_bert_predictions_and_metrics(
    model_key=model_key,
    results=results,
    test_data=test_data,
    task_name=TASK_NAME,
    save_dataframe_csv_func=save_dataframe_csv,
)


#### 4.2.5 MuRIL


In [ ]:
model_key = 'muril'
output_dir = results_dir / model_key

# Run training and evaluation
results = run_span_ner_model(
    model_key=model_key,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    output_dir=str(output_dir),
    **TRAINING_CONFIG,
    device=device,
)

# Store results
all_results[model_key] = results

# Save predictions and metrics using utility function
save_bert_predictions_and_metrics(
    model_key=model_key,
    results=results,
    test_data=test_data,
    task_name=TASK_NAME,
    save_dataframe_csv_func=save_dataframe_csv,
)


## 5. Results Comparison and Analysis


### 5.1 Load Previously Saved Results

If any models were run previously and saved, load them here. 


In [ ]:
# Load any previously saved results
print("Loading previously saved results...")
print("=" * 80)

model_names = ['gliner', 'confliBERT', 'deberta-v3', 'xlm-roberta', 'spanbert', 'muril']

loaded_count = 0
skipped_count = 0

for model_name in model_names:
    # Check if this model already has in-memory results
    if model_name in all_results and all_results[model_name] is not None:
        print(f"✓ {model_name}: Already in memory (from this session)")
        skipped_count += 1
        continue
    
    # Try to load from disk
    predictions_path = results_dir / f"{model_name}_predictions.csv"
    metrics_path = results_dir / f"{model_name}_metrics.csv"
    
    if predictions_path.exists() and metrics_path.exists():
        try:
            # Load predictions
            preds_df = pd.read_csv(predictions_path)
            
            # Load metrics
            metrics_df = pd.read_csv(metrics_path)
            
            # Reconstruct the metrics structure
            metrics = {
                'exact_match': metrics_df['exact_match'].iloc[0],
                'fuzzy_match': metrics_df['fuzzy_match'].iloc[0],
                'micro_precision': metrics_df.get('micro_precision', pd.Series([0])).iloc[0] if 'micro_precision' in metrics_df.columns else 0,
                'micro_recall': metrics_df.get('micro_recall', pd.Series([0])).iloc[0] if 'micro_recall' in metrics_df.columns else 0,
                'micro_f1': metrics_df['micro_f1'].iloc[0],
                'per_level': {
                    'state': {'f1': metrics_df['state_f1'].iloc[0]},
                    'district': {'f1': metrics_df['district_f1'].iloc[0]},
                    'village': {'f1': metrics_df['village_f1'].iloc[0]},
                    'other_locations': {'f1': metrics_df['other_locations_f1'].iloc[0]}
                }
            }
            
            # Store in all_results
            all_results[model_name] = {
                'metrics': metrics,
                'predictions': preds_df['prediction'].values.tolist() if 'prediction' in preds_df.columns else [],
                'ground_truth': preds_df['ground_truth'].values.tolist() if 'ground_truth' in preds_df.columns else []
            }
            
            print(f"✓ {model_name}: Loaded from disk")
            loaded_count += 1
        except Exception as e:
            print(f"✗ {model_name}: Error loading - {e}")
    else:
        print(f"✗ {model_name}: No saved results found")

print("=" * 80)
print(f"Summary: {skipped_count} in memory, {loaded_count} loaded from disk, {len(model_names) - skipped_count - loaded_count} missing")
print(f"Total available: {len([m for m in model_names if m in all_results and all_results[m] is not None])}/{len(model_names)} models")
print("=" * 80)


### 5.2 Compile All Results


In [ ]:
# Compile metrics from all models
all_metrics = []

for model_name, result in all_results.items():
    metrics = result['metrics'] if 'metrics' in result else result['test_metrics']
    
    metric_row = {
        'model': model_name,
        'exact_match': metrics['exact_match'],
        'fuzzy_match': metrics['fuzzy_match'],
        'micro_precision': metrics['micro_precision'],
        'micro_recall': metrics['micro_recall'],
        'micro_f1': metrics['micro_f1'],
        'state_f1': metrics['per_level']['state']['f1'],
        'district_f1': metrics['per_level']['district']['f1'],
        'village_f1': metrics['per_level']['village']['f1'],
        'other_locations_f1': metrics['per_level']['other_locations']['f1'],
    }
    all_metrics.append(metric_row)

# Create DataFrame
metrics_comparison = pd.DataFrame(all_metrics)

print("✅ Compiled metrics from all models")
print(f"Models evaluated: {len(all_metrics)}")


### 5.3 Overall Performance Comparison


In [ ]:
# Display overall metrics comparison
print("\n" + "="*100)
print("OVERALL PERFORMANCE COMPARISON")
print("="*100)

# Sort by micro F1
sorted_metrics = metrics_comparison.sort_values('micro_f1', ascending=False)

# Display table
display_cols = ['model', 'exact_match', 'fuzzy_match', 'micro_f1', 'micro_precision', 'micro_recall']
print("\n" + sorted_metrics[display_cols].to_string(index=False))
print("="*100)

# Highlight best model
best_model = sorted_metrics.iloc[0]
print(f"\n🏆 Best Model: {best_model['model']}")
print(f"   Micro F1: {best_model['micro_f1']:.4f}")
print(f"   Exact Match: {best_model['exact_match']:.4f}")
print(f"   Fuzzy Match: {best_model['fuzzy_match']:.4f}")


### 5.4 Per-Level Performance Comparison


In [ ]:
# Display per-level metrics
print("\n" + "="*100)
print("PER-LEVEL F1 SCORES")
print("="*100)

# Sort by micro F1
sorted_metrics = metrics_comparison.sort_values('micro_f1', ascending=False)

# Display table
level_cols = ['model', 'state_f1', 'district_f1', 'village_f1', 'other_locations_f1']
print("\n" + sorted_metrics[level_cols].to_string(index=False))
print("="*100)

# Find best model per level
print("\n🏆 Best Models per Level:")
for level in ['state', 'district', 'village', 'other_locations']:
    col = f'{level}_f1'
    best_idx = metrics_comparison[col].idxmax()
    best_model_name = metrics_comparison.loc[best_idx, 'model']
    best_score = metrics_comparison.loc[best_idx, col]
    print(f"   {level:20s}: {best_model_name:15s} (F1={best_score:.4f})")


### 5.5 Visualizations


In [ ]:
# Create comparison visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Overall Metrics (Exact, Fuzzy, Micro F1)
ax1 = axes[0, 0]
sorted_by_f1 = metrics_comparison.sort_values('micro_f1')
x_pos = np.arange(len(sorted_by_f1))
width = 0.25

ax1.barh(x_pos - width, sorted_by_f1['exact_match'], width, label='Exact Match', alpha=0.8)
ax1.barh(x_pos, sorted_by_f1['fuzzy_match'], width, label='Fuzzy Match', alpha=0.8)
ax1.barh(x_pos + width, sorted_by_f1['micro_f1'], width, label='Micro F1', alpha=0.8)

ax1.set_yticks(x_pos)
ax1.set_yticklabels(sorted_by_f1['model'])
ax1.set_xlabel('Score')
ax1.set_title('Overall Performance Comparison', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# 2. Per-Level F1 Scores
ax2 = axes[0, 1]
sorted_by_f1 = metrics_comparison.sort_values('micro_f1')
x_pos = np.arange(len(sorted_by_f1))
width = 0.2

ax2.barh(x_pos - 1.5*width, sorted_by_f1['state_f1'], width, label='State', alpha=0.8)
ax2.barh(x_pos - 0.5*width, sorted_by_f1['district_f1'], width, label='District', alpha=0.8)
ax2.barh(x_pos + 0.5*width, sorted_by_f1['village_f1'], width, label='Village', alpha=0.8)
ax2.barh(x_pos + 1.5*width, sorted_by_f1['other_locations_f1'], width, label='Other Locations', alpha=0.8)

ax2.set_yticks(x_pos)
ax2.set_yticklabels(sorted_by_f1['model'])
ax2.set_xlabel('F1 Score')
ax2.set_title('Per-Level F1 Scores', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

# 3. Micro F1 Bar Chart
ax3 = axes[1, 0]
sorted_by_f1 = metrics_comparison.sort_values('micro_f1')
colors = plt.cm.viridis(np.linspace(0, 1, len(sorted_by_f1)))

ax3.barh(sorted_by_f1['model'], sorted_by_f1['micro_f1'], color=colors, alpha=0.8)
ax3.set_xlabel('Micro F1 Score')
ax3.set_title('Micro F1 Ranking', fontsize=14, fontweight='bold')
ax3.grid(axis='x', alpha=0.3)

# Add value labels
for i, (idx, row) in enumerate(sorted_by_f1.iterrows()):
    ax3.text(row['micro_f1'], i, f" {row['micro_f1']:.3f}", va='center')

# 4. Precision vs Recall
ax4 = axes[1, 1]
colors_dict = {model: plt.cm.tab10(i) for i, model in enumerate(metrics_comparison['model'])}

for idx, row in metrics_comparison.iterrows():
    ax4.scatter(row['micro_recall'], row['micro_precision'], 
               s=200, alpha=0.6, 
               color=colors_dict[row['model']],
               label=row['model'])
    ax4.text(row['micro_recall'], row['micro_precision'], 
            row['model'], fontsize=8, ha='right', va='bottom')

ax4.set_xlabel('Micro Recall')
ax4.set_ylabel('Micro Precision')
ax4.set_title('Precision vs Recall', fontsize=14, fontweight='bold')
ax4.grid(alpha=0.3)
ax4.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

# Adjust layout
plt.tight_layout()

# Save figure
fig_path = results_dir / "bert_model_comparison.png"
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f"\n✅ Visualization saved to {fig_path}")

plt.show()


### 5.6 Save Combined Metrics and Predictions


In [ ]:
# Save combined metrics
metrics_path = save_dataframe_csv(
    metrics_comparison, 
    "location_bert_metrics_combined_overall.csv", 
    TASK_NAME
)
print(f"✅ Combined metrics saved to: {metrics_path}")

# Create per-level metrics DataFrame
per_level_records = []
for model_name, result in all_results.items():
    metrics = result['metrics'] if 'metrics' in result else result['test_metrics']
    
    for level in ['state', 'district', 'village', 'other_locations']:
        per_level_records.append({
            'model': model_name,
            'level': level,
            'precision': metrics['per_level'][level]['precision'],
            'recall': metrics['per_level'][level]['recall'],
            'f1': metrics['per_level'][level]['f1'],
        })

per_level_df = pd.DataFrame(per_level_records)
per_level_path = save_dataframe_csv(
    per_level_df,
    "location_bert_metrics_combined_levels.csv",
    TASK_NAME
)
print(f"✅ Per-level metrics saved to: {per_level_path}")

# Combine all predictions with incident_number and incident_summary
all_predictions = []
for model_name, result in all_results.items():
    predictions = result['predictions']
    ground_truth = result['ground_truth']
    
    for i, (pred, gt) in enumerate(zip(predictions, ground_truth)):
        all_predictions.append({
            'model': model_name,
            'example_idx': i,
            'incident_number': test_data[i]['metadata']['incident_number'],
            'incident_summary': test_data[i]['text'],
            'ground_truth': gt,
            'prediction': pred,
        })

predictions_df = pd.DataFrame(all_predictions)
predictions_path = save_dataframe_csv(
    predictions_df,
    "location_bert_predictions_combined.csv",
    TASK_NAME
)
print(f"✅ Combined predictions saved to: {predictions_path}")


### 5.7 Final Summary


In [ ]:
print("\n" + "="*100)
print("SPAN-BASED NER LOCATION EXTRACTION - FINAL SUMMARY")
print("="*100)

print(f"\n📊 Models Evaluated: {len(all_results)}")
print(f"   - GLiNER (zero-shot baseline)")
print(f"   - 5 BERT-based NER models")

print(f"\n📈 Best Overall Performance:")
best_model = metrics_comparison.sort_values('micro_f1', ascending=False).iloc[0]
print(f"   Model: {best_model['model']}")
print(f"   Micro F1: {best_model['micro_f1']:.4f}")
print(f"   Exact Match: {best_model['exact_match']:.4f}")
print(f"   Fuzzy Match: {best_model['fuzzy_match']:.4f}")

print(f"\n📁 Results Location: {results_dir}")
print(f"   - Individual model predictions and metrics")
print(f"   - Combined metrics CSV")
print(f"   - Combined predictions CSV")
print(f"   - Visualization plots")

print("\n✅ All models trained and evaluated successfully!")
print("="*100)
